**Versión 2 (14-nov-2026):** mismo análisis que la versión anterior, ejecutado de principio a fin, con resultados visibles, CSV de salida y versiones de las librerías registradas en la última celda.

# VitalDB: prueba de viabilidad para ChroniCare-AI (solo conteos)

**Autor:** Aly Gallo · **Fecha de la corrida:** se imprime en la primera celda

**Pregunta:** ¿VitalDB tiene suficientes pacientes con laboratorio *antes* de una cirugía y *después* de ella para probar si el estado biológico previo predice la recuperación?

**Lo que hace este cuaderno:** descarga los datos abiertos de VitalDB, describe la población, cuenta cuántos pacientes tienen cada prueba antes y después de la cirugía, mide los tiempos de las mediciones y compara ventanas de recuperación (W).

**Lo que NO hace:** no mira ningún desenlace de salud (mortalidad, complicaciones, estancia) ni calcula el BSI ni prueba ninguna hipótesis. Es solo contar.

**Datos:** Lee HC, Jung CW et al. VitalDB, a high-fidelity multi-parameter vital signs database in surgical patients. *Sci Data* 9, 279 (2022), https://doi.org/10.1038/s41597-022-01411-5. Conjunto de datos: Lee H, Jung C (2022), PhysioNet, https://doi.org/10.13026/czw8-9p62. Licencia CC BY 4.0.

**Registro de decisiones (para reproducibilidad):**
- Antes de la cirugía = `dt <= 0` (el tiempo cero es el inicio del registro del caso; los valores negativos son previos).
- Después de la cirugía = `dt >= opend` (desde el fin de la operación).
- La ventana W se elige por disponibilidad de mediciones y por la escala clínica (la PCR se resuelve en ~1 semana), **no** por trayectorias ni desenlaces.
- Las reoperaciones (mismo `subjectid`) se cuentan por cirugía aquí; el análisis posterior debe agrupar por paciente.

In [1]:
import sys, datetime, json
import numpy as np, pandas as pd
print("Fecha:", datetime.date.today(), "| Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

# Parametros (todos fijados aqui para que se vean)
PRUEBAS_INFLAMACION = ["crp", "alb", "wbc"]        # PCR, albumina, leucocitos
VENTANA_PREVIA_DIAS = 7                              # medicion previa: hasta 7 dias antes de la cirugia
VENTANAS_W = [5, 7, 10, 14]                          # dias despues de la cirugia que se comparan
MIN_MEDICIONES_POST = 3                              # minimo de mediciones dentro de W
URL_LABS  = "https://api.vitaldb.net/labs"
URL_CASES = "https://api.vitaldb.net/cases"


Fecha: 2026-11-17 | Python 3.13.18 | pandas 2.2.3 | numpy 2.4.6


## Paso 1. Descargar los datos
Si falla, el mensaje de error dice cual de las dos descargas no funciono.

In [2]:
lab = pd.read_csv(URL_LABS)
cases = pd.read_csv(URL_CASES)
print("lab:", lab.shape, "| cases:", cases.shape)
print("Esperado segun el articulo: 928448 filas de laboratorio y 6388 cirugias")
if lab.shape[0] != 928448 or cases.shape[0] != 6388:
    print("ATENCION: los tamanos no coinciden con los publicados; revisa la version de los datos.")
print("Columnas de lab:", lab.columns.tolist())
print(lab.head())


lab: (928448, 4) | cases: (6388, 74)
Esperado segun el articulo: 928448 filas de laboratorio y 6388 cirugias
Columnas de lab: ['caseid', 'dt', 'name', 'result']
   caseid      dt name  result
0       1  594470  alb     2.9
1       1  399575  alb     3.2
2       1   12614  alb     3.4
3       1  137855  alb     3.6
4       1  399575  alt    12.0


## Paso 2. Quien es la poblacion (sin ningun desenlace de salud)

In [3]:
print("Cirugias:", len(cases), "| pacientes unicos:", cases.subjectid.nunique(),
      "| reoperaciones:", len(cases) - cases.subjectid.nunique())
print("\nEmergencia (1) vs electiva (0):", cases.emop.value_counts().to_dict())
print("\nASA:", cases.asa.value_counts(dropna=False).sort_index().to_dict())
print("ASA >= 3 (%):", round((cases.asa >= 3).mean() * 100, 1))
print("\nDepartamento:", cases.department.value_counts().to_dict())
print("\nTipo de cirugia:", cases.optype.value_counts().to_dict())
print("\nDuracion de la operacion, mediana (min):", round(((cases.opend - cases.opstart) / 60).median(), 0))


Cirugias: 6388 | pacientes unicos: 6090 | reoperaciones: 298

Emergencia (1) vs electiva (0): {0: 5606, 1: 782}

ASA: {1.0: 1792, 2.0: 3699, 3.0: 703, 4.0: 48, 6.0: 13, nan: 133}
ASA >= 3 (%): 12.0

Departamento: {'General surgery': 4930, 'Thoracic surgery': 1111, 'Gynecology': 230, 'Urology': 117}

Tipo de cirugia: {'Colorectal': 1350, 'Biliary/Pancreas': 812, 'Others': 799, 'Stomach': 676, 'Major resection': 584, 'Minor resection': 553, 'Breast': 434, 'Transplantation': 403, 'Vascular': 262, 'Hepatic': 258, 'Thyroid': 257}

Duracion de la operacion, mediana (min): 110.0


## Paso 3. Cuantos pacientes tienen cada prueba antes y despues de la cirugia

In [4]:
d = lab.merge(cases[["caseid", "opend", "emop"]], on="caseid", how="left")
d["dias_post"] = (d.dt - d.opend) / 86400
print("dt minimo / maximo (dias):", round(lab.dt.min() / 86400, 1), round(lab.dt.max() / 86400, 1))
print("% de resultados con dt < 0:", round((lab.dt < 0).mean() * 100, 1))

tests = ["crp", "esr", "fib", "wbc", "alb", "gluc", "hb", "plt", "cr", "gfr", "tprot", "ast", "alt"]
rows = []
for t in tests:
    x = d[d.name == t]
    pre = set(x[x.dt <= 0].caseid)
    post = set(x[x.dt >= x.opend].caseid)
    n_post = x[x.dt >= x.opend].groupby("caseid").size()
    post2 = set(n_post[n_post >= 2].index)
    rows.append((t, len(pre), len(post), len(pre & post), len(pre & post2)))
tabla_conteo = pd.DataFrame(rows, columns=["prueba", "con_antes", "con_despues", "antes_y_despues", "antes_y_2_despues"])
print(tabla_conteo.to_string(index=False))


dt minimo / maximo (dias): -90.0 90.0
% de resultados con dt < 0: 26.2
prueba  con_antes  con_despues  antes_y_despues  antes_y_2_despues
   crp       3030         4184             2545               2255
   esr        455          250               44                 27
   fib       3617         3030             2345               1601
   wbc       4235         4843             3707               3226
   alb       4154         4644             3520               2993
  gluc       3771         3190             2253               1771
    hb       4235         4844             3708               3226
   plt       4227         4840             3698               3211
    cr       4110         4442             3382               3018
   gfr       4077         4428             3361               2996
 tprot       4154         4641             3517               2991
   ast       4157         4641             3519               2995
   alt       4157         4641             3519           

## Paso 4. Tiempos de las mediciones en la cohorte de inflamacion
Cohorte = cirugias con PCR, albumina y leucocitos antes y con 2 o mas mediciones despues.

In [5]:
def ok(t):
    x = d[d.name == t]
    pre = set(x[x.dt <= 0].caseid)
    post = x[x.dt >= x.opend].groupby("caseid").size()
    return pre & set(post[post >= 2].index)

coh = sorted(set.intersection(*[ok(t) for t in PRUEBAS_INFLAMACION]))
print("Cohorte (antes y >=2 despues):", len(coh), "| de emergencia:", int(cases[cases.caseid.isin(coh)].emop.sum()))

x = d[d.caseid.isin(coh) & d.name.isin(PRUEBAS_INFLAMACION)]
pre_dias = x[x.dt <= 0].groupby(["caseid", "name"]).dt.max().abs() / 86400
post = x[x.dt >= x.opend]
n_post = post.groupby(["caseid", "name"]).size()
span = post.groupby(["caseid", "name"]).dias_post.agg(lambda s: s.max() - s.min())
print("\nDIAS ENTRE LA ULTIMA MEDICION PREVIA Y LA CIRUGIA")
print(pre_dias.groupby("name").describe(percentiles=[.25, .5, .75, .9]).round(1).to_string())
print("\nNUMERO DE MEDICIONES DESPUES")
print(n_post.groupby("name").describe(percentiles=[.25, .5, .75]).round(1).to_string())
print("\nDIAS CUBIERTOS DESPUES (primera a ultima)")
print(span.groupby("name").describe(percentiles=[.25, .5, .75]).round(1).to_string())


Cohorte (antes y >=2 despues): 2202 | de emergencia: 326

DIAS ENTRE LA ULTIMA MEDICION PREVIA Y LA CIRUGIA
       count  mean   std  min  25%  50%  75%  90%   max
name                                                   
alb   2202.0   2.5   6.5  0.0  0.7  1.2  1.8  3.6  86.0
crp   2202.0   4.5  10.7  0.0  0.8  1.5  2.0  8.4  89.2
wbc   2202.0   2.3   6.2  0.0  0.7  1.1  1.8  2.9  89.2

NUMERO DE MEDICIONES DESPUES
       count  mean   std  min  25%  50%   75%    max
name                                                
alb   2202.0   7.3   9.1  2.0  3.0  4.0   7.0  100.0
crp   2202.0   6.3   6.4  2.0  3.0  4.0   7.0   54.0
wbc   2202.0   8.9  11.1  2.0  3.0  5.0  10.0  129.0

DIAS CUBIERTOS DESPUES (primera a ultima)
       count  mean   std  min  25%  50%   75%   max
name                                               
alb   2202.0  13.4  20.3  0.4  1.8  4.8  13.5  89.7
crp   2202.0  11.7  18.2  0.2  2.1  4.0  10.6  89.7
wbc   2202.0  14.4  20.8  0.2  2.7  5.5  14.4  89.7


## Paso 5. Elegir la ventana W
Regla: cuantos pacientes tienen, para las tres pruebas, una medicion previa en los 7 dias anteriores, al menos 3 mediciones dentro de W dias y la ultima medicion en la segunda mitad de la ventana.

In [6]:
def cuenta(W):
    sets = []
    for t in PRUEBAS_INFLAMACION:
        x = d[d.name == t]
        pre = set(x[(x.dt <= 0) & (x.dt >= -VENTANA_PREVIA_DIAS * 86400)].caseid)
        p = x[(x.dias_post >= 0) & (x.dias_post <= W)].groupby("caseid").dias_post.agg(["size", "max"])
        post = set(p[(p["size"] >= MIN_MEDICIONES_POST) & (p["max"] >= W / 2)].index)
        sets.append(pre & post)
    return len(set.intersection(*sets))

tabla_W = pd.DataFrame([(W, cuenta(W)) for W in VENTANAS_W], columns=["W_dias", "cirugias"])
print(tabla_W.to_string(index=False))
print("\nReferencia: la simulacion nula pedia >= 1000 a 2000 casos analizables para tener poder 0.80 con la medida 'fraccion de la caida recuperada'.")


 W_dias  cirugias
      5       643
      7       774
     10       665
     14       444

Referencia: la simulacion nula pedia >= 1000 a 2000 casos analizables para tener poder 0.80 con la medida 'fraccion de la caida recuperada'.


## Paso 6. Guardar los resultados
Los archivos quedan en la carpeta de archivos de Colab (icono de carpeta, a la izquierda). Descargalos o muevelos a tu Drive.

In [7]:
tabla_conteo.to_csv("conteo_por_prueba.csv", index=False)
tabla_W.to_csv("conteo_por_ventana.csv", index=False)
resumen = {
    "fecha": str(datetime.date.today()),
    "cirugias": int(len(cases)),
    "pacientes_unicos": int(cases.subjectid.nunique()),
    "cohorte_PCR_alb_wbc_antes_y_2_despues": int(len(coh)),
    "cirugias_por_ventana": {int(a): int(b) for a, b in zip(tabla_W.W_dias, tabla_W.cirugias)},
    "reglas": {"pruebas": PRUEBAS_INFLAMACION, "previa_dias": VENTANA_PREVIA_DIAS,
               "min_mediciones_post": MIN_MEDICIONES_POST}
}
with open("resumen_viabilidad.json", "w") as f:
    json.dump(resumen, f, indent=2)
print(json.dumps(resumen, indent=2))


{
  "fecha": "2026-11-17",
  "cirugias": 6388,
  "pacientes_unicos": 6090,
  "cohorte_PCR_alb_wbc_antes_y_2_despues": 2202,
  "cirugias_por_ventana": {
    "5": 643,
    "7": 774,
    "10": 665,
    "14": 444
  },
  "reglas": {
    "pruebas": [
      "crp",
      "alb",
      "wbc"
    ],
    "previa_dias": 7,
    "min_mediciones_post": 3
  }
}


## Lectura (corrida del 27-oct-2026)
- **Resultado principal:** con W = 7 dias hay **774 cirugias** con PCR, albumina y leucocitos antes y despues (643 con W = 5, 665 con W = 10, 444 con W = 14).
- **Tamano:** es menos de las 1000 a 2000 que pedia la simulacion nula. VitalDB sirve como ensayo general y no como base de la prueba confirmatoria.
- **Tiempos:** la medicion previa esta a una mediana de 1 a 1.5 dias de la cirugia; las mediciones posteriores cubren una mediana de 4 a 5.5 dias (la estancia hospitalaria).
- **Cohorte amplia:** 2202 cirugias con las tres pruebas antes y al menos 2 despues, de ellas 326 de emergencia.
- Esto **no** prueba la hipotesis; solo dice si hay datos suficientes y como medir.
- **Limites a declarar:** poblacion coreana de un solo hospital, mayoria ASA 1 o 2, sin lipidos ni HbA1c ni diferencial de linfocitos, medicion informativa (la PCR se pide a quien tiene motivo), seguimiento que termina con el alta y 298 reoperaciones que deben agruparse por paciente.

## Cierre: guardar salidas y versiones
Esta celda final escribe `requirements_VitalDB_viabilidad_ChroniCare.txt` (versiones usadas), calcula la huella sha256 de cada CSV y descarga todo en un .zip. Ejecútala al final, después de las demás.

In [8]:
import sys, os, zipfile, hashlib, importlib
from google.colab import files

NB = "VitalDB_viabilidad_ChroniCare_v2"
CSVS = ['conteo_por_prueba.csv', 'conteo_por_ventana.csv']

versiones = {"python": sys.version.split()[0]}
for m in ["numpy", "pandas", "scipy", "statsmodels", "matplotlib"]:
    try:
        versiones[m] = importlib.import_module(m).__version__
    except Exception:
        pass

req = "requirements_" + NB + ".txt"
with open(req, "w") as f:
    f.write("# python " + versiones["python"] + "\n")
    for k, v in versiones.items():
        if k != "python":
            f.write(k + "==" + v + "\n")
print("Versiones:", versiones)

faltan = [c for c in CSVS if not os.path.exists(c)]
if faltan:
    print("ATENCION: no se encontraron estos CSV (corre antes todas las celdas):", faltan)

huellas = {}
for c in CSVS:
    if os.path.exists(c):
        huellas[c] = hashlib.sha256(open(c, "rb").read()).hexdigest()
print("Huellas sha256:")
for k, v in huellas.items():
    print(" ", k, v[:16] + "...")

zipname = "salidas_" + NB + ".zip"
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for c in CSVS:
        if os.path.exists(c):
            z.write(c)
    z.write(req)
print("Listo:", zipname)
files.download(zipname)


Versiones: {'python': '3.13.18', 'numpy': '2.4.6', 'pandas': '2.2.3', 'scipy': '1.16.3', 'statsmodels': '0.15.0', 'matplotlib': '3.10.0'}
Huellas sha256:
  conteo_por_prueba.csv b64179873d55b986...
  conteo_por_ventana.csv 88dc921e423388b4...
Listo: salidas_VitalDB_viabilidad_ChroniCare_v2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>